# Experimentos de ranking

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve().parent  # if notebook is in notebooks/
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
from src.paths import PROC_DIR
from src.retrieval import BM25Retriever
from src.evaluation import map_relevance, evaluate_retrieval, evaluate_ranking
from src.features import build_ranking_features
from src.ranking import train_ranker, score_candidates
from src.evaluation import evaluate_ranking


In [4]:
TOP_K = 100

In [5]:
products = pd.read_parquet(PROC_DIR / "products.parquet")
queries = pd.read_parquet(PROC_DIR / "queries.parquet")
labels = pd.read_parquet(PROC_DIR / "labels.parquet")
splits = pd.read_parquet(PROC_DIR / "splits.parquet")

In [6]:
labels = labels.copy()
labels["relevance"] = map_relevance(labels["label"])

train_qids = set(splits.loc[splits["split"] == "train", "query_id"])
val_qids = set(splits.loc[splits["split"] == "val", "query_id"])

train_queries = queries[queries["query_id"].isin(train_qids)]
val_queries = queries[queries["query_id"].isin(val_qids)]

train_labels = labels[labels["query_id"].isin(train_qids)]
val_labels = labels[labels["query_id"].isin(val_qids)]

### BM25

In [7]:
retriever = BM25Retriever().fit(products)
train_candidates = retriever.retrieve(train_queries, top_k=TOP_K)
val_candidates = retriever.retrieve(val_queries, top_k=TOP_K)

In [8]:
retrieval_metrics = evaluate_retrieval(val_candidates, val_labels, ks=[20, 50, TOP_K])
print("Retrieval metrics (val):", retrieval_metrics)

Retrieval metrics (val): {'recall@20': 0.12425121013626411, 'recall@50': 0.2508804523216268, 'recall@100': 0.3830537592768249}


In [9]:
train_feat = build_ranking_features(
    candidates=train_candidates,
    queries=train_queries,
    products=products,
    labels=train_labels,
)

In [10]:
val_feat = build_ranking_features(
    candidates=val_candidates,
    queries=val_queries,
    products=products,
    labels=val_labels,
)

In [11]:
print("Train shape:", train_feat.shape, "| Val shape:", val_feat.shape)
print("Val relevance counts inside top-K:\n", val_feat["relevance"].value_counts().sort_index())
print("Val candidates per query summary:\n", val_feat.groupby("query_id").size().describe())

Train shape: (33600, 31) | Val shape: (7200, 31)
Val relevance counts inside top-K:
 relevance
0    3086
1    2802
2    1312
Name: count, dtype: int64
Val candidates per query summary:
 count     72.0
mean     100.0
std        0.0
min      100.0
25%      100.0
50%      100.0
75%      100.0
max      100.0
dtype: float64


In [12]:
bm25_val_scored = val_feat[["query_id", "bm25_score", "relevance"]].rename(
    columns={"bm25_score": "score"}
)

In [13]:
bm25_ranking_metrics = evaluate_ranking(bm25_val_scored, ks=[10])

In [14]:
print("BM25-only ranking metrics (val):", bm25_ranking_metrics)

BM25-only ranking metrics (val): {'ndcg@10': 0.7102650918215304}
